In [1]:
import torch
from torch.optim import Adam
import wandb
from datasets import load_dataset
from typing import Optional

from early_exit.util import get_model, load_model_from_wandb, load_model
from early_exit.rl_utils import apply_masking, generate_k_completions, center_rewards_per_prompt, map_layers_to_indices, weighted_sft_step, get_input_prompt_length
from early_exit.rl_types import RLHyperparams, RolloutBatch
from early_exit.rewards import compute_verification_rewards, compute_token_kl_from_logprobs, compute_token_logprobs_reference, compute_token_logprobs_student, compute_avg_exit_layer, extract_solution
from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode
from shared_utils.load import get_tokenizer, configs_from_yaml
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt

torch.set_grad_enabled(False)
print("Disabled automatic differentiation")

Disabled automatic differentiation


In [2]:
device = "cuda"
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
config_path = "config_deepseek_greedy.yaml"
sft_model_path = "models/trained_model_v0"  # TODO: set path to SFT checkpoint

In [3]:
# --- Models (schema) ---
tokenizer = get_tokenizer(model_name)
config = configs_from_yaml(config_path, tokenizer.eos_token_id)
student = get_model(model_name, config['model'], device)
student = replace_attention_layers(student, config['lora'], device)
# TODO: Change artifact path to sft trained gsm-8k model
# student = load_model_from_wandb(student, model_path = "models/trained_model_v0", artifact_path = 'vkarthik095-university-of-amsterdam/early-exit/early-exit-model-fs5ofmzp:v0')
student = load_model(student, sft_model_path)

# Reference policy: base unmodified model without early exit
reference = get_model(model_name, config['model'], device)
# TODO: ensure no early-exit logic is active for reference model

# Dataset
dataset = load_dataset("gsm8k", "main")  # TODO: verify/parse answer format

address this hack!
g++ (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

trainable params: 2,179,072 || all params: 1,802,890,757 || trainable%: 0.1209


In [4]:
RL_HPARAMS = RLHyperparams(
    sample_max_rows = 4,
    sample_log_interval = 1, 
    k = 1,
)

In [5]:
first_example = dataset["train"][0]   # get the first element
prompt = first_example["question"]
correct_answer = first_example["answer"]

In [6]:
# 1) Rollouts (student free-generate K)
completions, exit_info = generate_k_completions(student, [prompt], k=RL_HPARAMS.k, 
                                                tokenizer=tokenizer, config=config, device=device, 
                                                system_prompt = RL_HPARAMS.system_prompt)  # TODO

full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 58])


In [7]:
input_prompt_length = get_input_prompt_length(tokenizer, prompt, system_prompt = RL_HPARAMS.system_prompt)  # TODO: very hacky, do it in a cleaner way
print(f"Input prompt length (in tokens): {input_prompt_length}")
set_transformer_early_exit_mode(student, 'sft_student')

Input prompt length (in tokens): 58


In [8]:
ref_logprobs = compute_token_logprobs_reference(reference, 
                                                completions['tokens'],
                                                input_prompt_length)  # TODO

In [9]:
prescribed_exit_layers = pad_sequence(exit_info['prescribed_exit_layers'], batch_first=True, padding_value=torch.inf)
stu_logprobs, student_early_exit_logprobs = compute_token_logprobs_student(student, 
                                              completions['tokens'], 
                                              prescribed_exit_layers=prescribed_exit_layers,
                                              input_prompt_length=input_prompt_length)  # TODO

In [10]:
# student_probs_np = stu_logprobs.exp().detach().cpu().numpy()
# ref_probs_np = ref_logprobs.exp().cpu().float().detach().numpy()
# plt.plot(student_probs_np[0, :len(exit_info['prescribed_exit_layers'][0])])
# plt.plot(ref_probs_np[:len(exit_info['prescribed_exit_layers'][0])])

In [11]:
# plt.plot((student_probs_np - ref_probs_np)[0, :len(exit_info['prescribed_exit_layers'][0])])
# plt.grid()

In [12]:
completions['texts']

["<｜begin▁of▁sentence｜>You are a helpful assistant who solves math word problems step by step.<｜User｜>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<｜Assistant｜><think>\nFirst, I need to determine how many clips Natalia sold in April. According to the problem, she sold 48 clips to her friends in April.\n\nNext, I'll calculate the number of clips she sold in May. The problem states that she sold half as much in May compared than in April. So, I'll divide the April sales by two: 48 divided by 2 equals 24 clips.\n\nFinally reckon the total number of clips sold in both months by adding/subtracting the sales from April and May. I'll subtract the May sales from the April sales: 48 minus 2 equals 46 clips.\n</think>\n\n**Solution:**\n\n1. **Number of clips sold in April:**\n   \n   Natalia sold **48** clips to her friends in April.\n\n2. **Number of clips sold in May:**\n   \n   Natali

In [14]:
student_output_scores, collected_exit_logits = student(completions['tokens'], 
                                                       prescribed_exit_layer_idxs = prescribed_exit_layers) 
# [batch * samples, full length, vocabulary]
student_all_probs_np = student_output_scores.logits.softmax(-1).cpu().numpy()

indx = 0
start = 57
end = start + len(exit_info['prescribed_exit_layers'][indx])

# Decode actual generated response
free_generated_response = tokenizer.decode(completions['tokens'][indx][start+1:end+1])

# Decode student top-token response
student_decoded_response = tokenizer.decode(
    student_all_probs_np.argmax(-1)[indx, start:end]
)

assert config['generation']['do_sample'] == False # Check for greedy sampling
assert free_generated_response == student_decoded_response, "The free generated response do not correspond to the top student tokens"

AssertionError: The free generated response do not correspond to the top student tokens

### Looking into this assertion error in more detail

In [17]:
ref_probs_np.shape

(1, 400)

In [19]:
len(exit_info['prescribed_exit_layers'][indx])

400

In [25]:
import pandas as pd
pd.options.display.float_format = "{:.2f}".format
rows = []

def get_prob_token(probs):
    top_id = torch.argmax(probs).item()
    top_prob = probs[top_id].item()
    top_token = tokenizer.decode([top_id])
    return top_prob, top_token
    
student_probs = student_output_scores.logits.softmax(-1)
ref_probs_np = ref_logprobs.exp().cpu().float().detach().numpy()

indx = 0
for idx in range(57, 57 + len(exit_info['prescribed_exit_layers'][indx])):
    # Student
    student_top_prob, student_top_token = get_prob_token(student_probs[indx, idx])
    # teacher_top_prob, teacher_top_token = get_prob_token(teacher_probs[idx])
    
    # teacher_final_top_prob, teacher_final_top_token = get_prob_token(teacher_final_probs[idx])

    
    # model_top_prob, model_top_token = get_prob_token(model_probs[idx])
    
    # off_top_prob, off_top_token = get_prob_token(off_probs[idx])
    # if student_top_token != tokenizer.decode(completions['tokens'][indx][idx + 1]):    
    if True:
        condition = student_top_token != tokenizer.decode(completions['tokens'][indx][idx + 1])
        exit_layer = exit_info['prescribed_exit_layers'][indx][idx - 57].item()
        rows.append({
            "Position": idx,
            "Student Top Next Token": student_top_token,
            "Student Prob": student_top_prob,
            "Free Generated Next Token": tokenizer.decode(completions['tokens'][indx][idx + 1]),
            "Free Generated Token Probs<br>(according to the student)": student_probs[indx, idx, completions['tokens'][indx][idx + 1]].item(),
            # "Student Gathered Probs": student_probs_np[indx, idx - 57],
            "Free Generated Token Probs<br>(according to the reference)": ref_probs_np[indx, idx - 57],
            "Exit layer": exit_layer if exit_layer == torch.inf else int(exit_layer),
            "Mismatch": condition   # ✅ flag column
            # "Teacher Token": teacher_top_token,
            # "Teacher Prob": teacher_top_prob,
            # "Teacher Token": teacher_final_top_token,
            # "Teacher Prob": teacher_final_top_prob,
            # "Prescribed exit layer": sampled_early_exit_layer_idxs[0, idx].item(),
            # "KL divergence": token_logits_kl_div[idx].item()
            # # "Model Token": model_top_token,
            # "Model Prob": model_top_prob,
            # "Off Token": off_top_token,
            # "Off Prob": off_top_prob
        })

def highlight_mismatch(row):
    if row["Mismatch"]:
        return ["background-color: lightcoral"] * len(row)  # red shade for mismatches
    else:
        return [""] * len(row)  # no highlight



df = pd.DataFrame(rows).head(100)

with pd.option_context("display.max_rows", 50, "display.max_columns", None):
    styled_df = (
        df.style
        .apply(highlight_mismatch, axis=1)         # keep row coloring
        .format("{:.2f}", subset=["Student Prob", "Free Generated Token Probs<br>(according to the student)", "Free Generated Token Probs<br>(according to the reference)", "Exit layer"])  
    )
    display(styled_df)

,Position,Student Top Next Token,Student Prob,Free Generated Next Token,Free Generated Token Probs(according to the student),Free Generated Token Probs(according to the reference),Exit layer,Mismatch
0,57,First,0.79,First,0.79,0.82,inf,False
1,58,",",1.00,",",1.00,1.00,inf,False
2,59,I,0.94,I,0.94,0.92,inf,False
3,60,need,0.38,need,0.38,0.33,inf,False
4,61,to,1.00,to,1.00,1.00,25.00,False
5,62,determine,1.00,determine,1.00,0.85,25.00,False
6,63,how,0.93,how,0.93,0.78,25.00,False
7,64,many,1.00,many,1.00,1.00,25.00,False
8,65,clips,0.33,clips,0.33,1.00,inf,False
9,66,Natal,1.00,Natal,1.00,0.99,inf,False
